# Lokales Test-Notebook

Dieses Notebook lädt ein Kontaktdaten-Dataset, erzeugt sichere dynamische Test-E-Mails mit einem lokalen LLM und bewertet sie mit dem vorhandenen Spam-Modell.

In [4]:
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Protocol, Sequence, Tuple
import csv
import json
import os
import random
import re
from datetime import datetime, timezone

import requests
import kagglesdk.kaggle_env as kaggle_env

def get_web_endpoint(*args, **kwargs):
    return 'https://www.kaggle.com'

kaggle_env.get_web_endpoint = get_web_endpoint

import kagglehub
import kagglehub.handle as kagglehub_handle
kagglehub_handle.get_web_endpoint = get_web_endpoint

random.seed()

## 1. Setup

Imports und Notebook-Startkonfiguration.

In [5]:
DATASET_PATH = kagglehub.dataset_download('leadsblue/b2b-contact-database-sample-dataset')
print('Path to dataset files:', DATASET_PATH)

def list_dataset_files(dataset_path: str) -> List[str]:
    discovered_files = []
    for path in sorted(Path(dataset_path).rglob('*')):
        if path.is_file():
            discovered_files.append(str(path))
    return discovered_files


dataset_files = list_dataset_files(DATASET_PATH)
dataset_files[:20]

Path to dataset files: /home/schuetzj/.cache/kagglehub/datasets/leadsblue/b2b-contact-database-sample-dataset/versions/3


['/home/schuetzj/.cache/kagglehub/datasets/leadsblue/b2b-contact-database-sample-dataset/versions/3/globalb2bdataset.csv']

## 2. Dataset

Das Kaggle-Dataset wird geladen und die enthaltenen Dateien werden aufgelistet.

### 2.1 Generator, Parser und Link-Injektion

Dieser Block enthält die komplette Erzeugung der sicheren Test-E-Mails, die LLM-Parsing-Logik und die Link-Injektion. Der Inhalt bleibt unverändert, nur die Struktur wird klarer.

In [6]:
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Protocol, Sequence
import csv
import json
import os
import random
import re

import requests
import kagglesdk.kaggle_env as kaggle_env


def get_web_endpoint(*args, **kwargs):
    return 'https://www.kaggle.com'


kaggle_env.get_web_endpoint = get_web_endpoint

import kagglehub
import kagglehub.handle as kagglehub_handle
kagglehub_handle.get_web_endpoint = get_web_endpoint

random.seed()


@dataclass
class SyntheticMessage:
    text: str
    metadata: Dict[str, Any] = field(default_factory=dict)


class MessageGenerator(Protocol):
    def generate(self, count: int = 10) -> List[SyntheticMessage]:
        ...


class Classifier(Protocol):
    def predict(self, messages: Sequence[str]) -> List[Dict[str, Any]]:
        ...


def load_contact_rows(dataset_csv: str) -> List[Dict[str, str]]:
    with Path(dataset_csv).open(newline='', encoding='utf-8', errors='replace') as handle:
        reader = csv.DictReader(handle)
        return list(reader)


@dataclass
class SyntheticEmailDraft:
    to_name: str
    to_email: str
    subject: str
    body: str
    provider: str
    round_index: int
    metadata: Dict[str, Any] = field(default_factory=dict)


class EmailTextGenerator(Protocol):
    enabled: bool

    def generate(self, prompt: str) -> str:
        ...


class OllamaTextGenerator:
    def __init__(self, model: str = 'llama3.1:8b', base_url: str | None = None) -> None:
        self.model = model
        self.base_url = (base_url or os.getenv('OLLAMA_BASE_URL') or 'http://localhost:11434').rstrip('/')
        self.enabled = self._probe()

    def _probe(self) -> bool:
        try:
            response = requests.get(f'{self.base_url}/api/tags', timeout=2)
            return response.ok
        except requests.RequestException:
            return False

    def generate(self, prompt: str) -> str:
        if not self.enabled:
            raise RuntimeError('Ollama is not available')
        payload = {
            'model': self.model,
            'messages': [
                {'role': 'system', 'content': 'You create safe synthetic email drafts for classifier testing. Return only JSON.'},
                {'role': 'user', 'content': prompt},
            ],
            'stream': False,
            'format': 'json',
            'options': {
                'temperature': 0.5,
                'num_predict': 512,
            },
        }
        try:
            response = requests.post(f'{self.base_url}/api/chat', json=payload, timeout=120)
            response.raise_for_status()
        except requests.RequestException as exc:
            raise RuntimeError(f'Ollama request failed: {exc}') from exc

        response_payload = response.json()
        content = response_payload.get('message', {}).get('content', '')
        if not content:
            raise RuntimeError('Ollama returned no content')
        return str(content).strip()


def select_text_generator() -> OllamaTextGenerator:
    provider = OllamaTextGenerator()
    if not provider.enabled:
        raise RuntimeError('Ollama is not available; generation requires a running local Ollama instance')
    return provider


def _feedback_guidance(feedback: Dict[str, Any] | None) -> str:
    if not feedback:
        return 'Use a varied but neutral business email style.'
    suspicious_rate = float(feedback.get('suspicious_rate', 0.0))
    benign_rate = float(feedback.get('benign_rate', 0.0))
    if suspicious_rate >= 0.5:
        return 'Reduce spammy markers. Keep the content clearly benign, but vary structure, line breaks, and subject length.'
    if benign_rate >= 0.7:
        return 'Increase variety in layout, sentence length, and sign-off style while staying clearly non-malicious.'
    return 'Mix concise and verbose formats, alternate greetings, and vary the tone slightly.'


def _build_generation_prompt(row: Dict[str, str], round_index: int, feedback: Dict[str, Any] | None) -> str:
    decision_maker = (row.get('Decision Maker Name') or 'there').strip() or 'there'
    title = (row.get('Decision Maker Title') or 'decision maker').strip() or 'decision maker'
    industry = (row.get('Industry') or 'your sector').strip() or 'your sector'
    country = (row.get('Country') or 'your region').strip() or 'your region'
    contact_email = (row.get('Email Address') or '').strip()
    feedback_text = json.dumps(feedback or {}, ensure_ascii=False)
    guidance = _feedback_guidance(feedback)
    return (
        'You are a strict, rule-following AI generating highly realistic, synthetic business emails.\n\n'
        'CRITICAL RULES - READ CAREFULLY:\n'
        '1. MANDATORY LINK: You MUST include the exact text "[link]" somewhere naturally inside the "body".\n'
        '2. NO LINKS IN SUBJECT: The "subject" MUST NOT contain "[link]".\n'
        '3. NO OTHER PLACEHOLDERS: You are STRICTLY FORBIDDEN from using standard placeholders like [Company], [Your Name], [Contact Name], or [Insert Here]. Instead, invent realistic fake company names and sign off with a fictional human name.\n'
        '4. DO NOT COPY EXAMPLES: Generate a completely unique subject and body every time. Do not use the words from the JSON schema example.\n'
        '5. FORMAT: Return ONLY valid JSON. No markdown tags (like ```json), no conversational text.\n\n'
        'JSON SCHEMA REQUIRED:\n'
        '{\n'
        '  "subject": "<generate a unique, highly relevant subject line here without any links>",\n'
        '  "body": "<generate the email body here, ensuring you address the contact naturally and include the [link] placeholder exactly once>",\n'
        '  "tags": ["safe", "synthetic"]\n'
        '}\n\n'
        'TARGET CONTEXT & INPUTS:\n'
        f'- Contact Name: {decision_maker}\n'
        f'- Job Title: {title}\n'
        f'- Industry: {industry}\n'
        f'- Country: {country}\n'
        f'- Contact Email: {contact_email}\n'
        f'- Iteration Round: {round_index + 1}\n'
        f'- Previous Feedback: {feedback_text}\n'
        f'- Guidance/Prompting: {guidance}\n\n'
        'Return ONLY the JSON object now.'
    )


def _parse_llm_json(raw_text: str) -> Dict[str, Any]:
    start = raw_text.find('{')
    end = raw_text.rfind('}')
    if start == -1 or end == -1 or end <= start:
        raise ValueError('LLM response did not contain JSON')
    return json.loads(raw_text[start:end + 1])


def _contains_unsafe_markers(text: str) -> bool:
    normalized_text = text.lower()
    if normalized_text.count('[link]') != 1:
        return True
    forbidden_patterns = [
        r'\[(company|your name|contact name|insert here|placeholder|name|email|url|website)\]',
        r'\{[^}]+\}',
        r'```',
    ]
    return any(re.search(pattern, text, flags=re.IGNORECASE) for pattern in forbidden_patterns)


def _fallback_draft(row: Dict[str, str], round_index: int, feedback: Dict[str, Any] | None) -> SyntheticEmailDraft:
    raise RuntimeError('Fallback generation is disabled; Ollama must be available')


class DynamicSafeEmailGenerator:
    def __init__(self, dataset_csv: str, provider: EmailTextGenerator | None = None) -> None:
        self.dataset_csv = dataset_csv
        self.provider = provider or select_text_generator()
        self.feedback: Dict[str, Any] | None = None
        self.round_index = 0
        self.rows = load_contact_rows(dataset_csv)

    def update_feedback(self, feedback: Dict[str, Any]) -> None:
        self.feedback = feedback
        self.round_index += 1

    def _pick_rows(self, count: int) -> List[Dict[str, str]]:
        if not self.rows:
            return []
        if count >= len(self.rows):
            selection = list(self.rows)
            random.shuffle(selection)
            return selection
        return random.sample(self.rows, count)

    def _generate_one(self, row: Dict[str, str]) -> SyntheticEmailDraft:
        prompt = _build_generation_prompt(row, self.round_index, self.feedback)
        last_error = None
        for _ in range(2):
            try:
                raw_text = self.provider.generate(prompt)
                payload = _parse_llm_json(raw_text)
                subject = str(payload.get('subject', '')).strip()
                body = str(payload.get('body', '')).strip()
                tags = payload.get('tags', [])
                if not subject or not body:
                    last_error = 'missing_subject_or_body'
                    continue
                combined_text = f'{subject}\n\n{body}'
                if _contains_unsafe_markers(combined_text):
                    last_error = 'unsafe_markers_detected'
                    continue
                return SyntheticEmailDraft(
                    to_name=(row.get('Decision Maker Name') or 'there').strip() or 'there',
                    to_email=(row.get('Email Address') or '').strip(),
                    subject=subject,
                    body=body,
                    provider=type(self.provider).__name__,
                    round_index=self.round_index,
                    metadata={
                        'tags': tags,
                        'feedback': self.feedback or {},
                        'prompt_round': self.round_index + 1,
                    },
                )
            except Exception as exc:
                last_error = str(exc)
        raise RuntimeError(f'Ollama generation failed: {last_error or "unknown_llm_error"}')

    def generate(self, count: int = 10) -> List[SyntheticMessage]:
        rows = self._pick_rows(count)
        drafts: List[SyntheticEmailDraft] = [self._generate_one(row) for row in rows]
        messages: List[SyntheticMessage] = []
        for draft in drafts:
            messages.append(
                SyntheticMessage(
                    text=f'Subject: {draft.subject}\n\n{draft.body}',
                    metadata={
                        'to_name': draft.to_name,
                        'to_email': draft.to_email,
                        'provider': draft.provider,
                        'round_index': draft.round_index,
                        **draft.metadata,
                    },
                )
            )
        return messages


def generate_dynamic_test_emails(dataset_csv: str, limit: int = 10, provider: EmailTextGenerator | None = None, feedback: Dict[str, Any] | None = None) -> List[SyntheticEmailDraft]:
    generator = DynamicSafeEmailGenerator(dataset_csv, provider=provider)
    generator.feedback = feedback
    generated_messages = generator.generate(limit)
    drafts: List[SyntheticEmailDraft] = []
    for message in generated_messages:
        drafts.append(
            SyntheticEmailDraft(
                to_name=message.metadata.get('to_name', ''),
                to_email=message.metadata.get('to_email', ''),
                subject=message.text.split('\n\n', 1)[0].replace('Subject: ', '', 1),
                body=message.text.split('\n\n', 1)[1] if '\n\n' in message.text else '',
                provider=str(message.metadata.get('provider', 'ollama')),
                round_index=int(message.metadata.get('round_index', 0)),
                metadata=message.metadata,
            )
        )
    return drafts


def preview_email_drafts(drafts: Sequence[SyntheticEmailDraft], limit: int = 3) -> List[Dict[str, str]]:
    preview = []
    for draft in drafts[:limit]:
        preview.append({
            'to_name': draft.to_name,
            'to_email': draft.to_email,
            'subject': draft.subject,
            'body': draft.body,
            'provider': draft.provider,
        })
    return preview


contact_csv_path = next(path for path in dataset_files if path.endswith('.csv'))
contact_rows = load_contact_rows(contact_csv_path)
email_drafts = generate_dynamic_test_emails(contact_csv_path, limit=5)


malicious_looking_urls = [
    "https://www.g00gle.com/login",                 # Visual Spoofing (Nullen statt O)
    "https://www.exampIe.com/secure-auth",          # Visual Spoofing (Großes i statt kleines L)
    "http://update-billing.paypal-secure.net.co/",  # TLD-Manipulation & Subdomain-Spoofing
    "https://microsoft-support.local-admin.xyz/",   # Ungewöhnliche TLD (.xyz)
    "http://192.168.1.100/admin_portal/reset",      # Direkte IP-Adresse (oft verdächtig in E-Mails)
    "https://account-verify.amazon.com-login.info"  # Angehängte Wörter an echte Marken
]

def inject_test_links(drafts: List[SyntheticEmailDraft], url_list: List[str]) -> List[SyntheticEmailDraft]:
    """
    Geht durch alle E-Mail-Entwürfe und ersetzt den exakten String '[link]'
    im Body durch eine zufällige URL aus der übergebenen Liste.
    """
    for draft in drafts:
        if "[link]" in draft.body:
            selected_url = random.choice(url_list)
            draft.body = draft.body.replace("[link]", selected_url)
            draft.metadata['injected_url'] = selected_url
    return drafts


email_drafts_with_links = inject_test_links(email_drafts, malicious_looking_urls)
final_preview = preview_email_drafts(email_drafts_with_links, limit=5)
print(json.dumps(final_preview, indent=2, ensure_ascii=False))


class LLMClassifier:
    def __init__(self, provider: EmailTextGenerator):
        self.provider = provider

    def predict(self, messages: Sequence[str]) -> List[Dict[str, Any]]:
        prompt = (
            'Classify each email as spam or ham\n'
            'Return ONLY valid JSON as a list of objects with keys index, label, rationale.\n'
            'Links are a strong spam indicator, but consider the overall context. Be cautious with emails that have suspicious formatting, urls or content.\n'
            'Labels allowed: spam, ham\n\n'
            'Emails:\n'
            + '\n\n'.join(f'[{index}]\n{message}' for index, message in enumerate(messages))
        )
        try:
            raw_text = self.provider.generate(prompt)
            payload = _parse_llm_json(raw_text)
            items = payload if isinstance(payload, list) else payload.get('items', []) if isinstance(payload, dict) else []
            indexed = {int(item.get('index', position)): item for position, item in enumerate(items)}
            predictions: List[Dict[str, Any]] = []
            for index in range(len(messages)):
                item = indexed.get(index, {})
                label = str(item.get('label', 'ham')).strip().lower()
                if label == 'spam':
                    label = 'susp'
                elif label not in {'ham', 'susp'}:
                    label = 'ham'
                predictions.append({
                    'prediction': label,
                    'rationale': str(item.get('rationale', '')).strip(),
                    'raw': raw_text,
                })
            if len(predictions) < len(messages):
                predictions.extend([{'prediction': 'ham', 'rationale': 'missing_result', 'raw': raw_text}] * (len(messages) - len(predictions)))
            return predictions
        except Exception as exc:
            return [
                {
                    'prediction': 'ham',
                    'rationale': f'classification_failed: {exc}',
                    'raw': '',
                }
                for _ in messages
            ]


def classify_messages(classifier: LLMClassifier, messages: Sequence[SyntheticMessage]) -> List[Dict[str, Any]]:
    texts = [message.text for message in messages]
    predictions = classifier.predict(texts)
    for message, prediction in zip(messages, predictions):
        prediction['metadata'] = message.metadata
    return predictions


from copy import deepcopy


def score_feedback(predictions: Sequence[Dict[str, Any]]) -> Dict[str, Any]:
    suspicious = 0
    benign = 0

    for item in predictions:
        label = str(item.get('prediction', '')).lower()
        if label in {'spam', 'susp'}:
            suspicious += 1
        else:
            benign += 1

    total = max(len(predictions), 1)
    return {
        'total': len(predictions),
        'suspicious_rate': suspicious / total,
        'benign_rate': benign / total,
    }


def _render_round_table(round_summary: Dict[str, Any]) -> str:
    headers = ['Idx', 'To', 'Label', 'Susp', 'Ben']
    rows = []
    for index, (draft, prediction) in enumerate(zip(round_summary['drafts'], round_summary['predictions']), start=1):
        feedback = round_summary['feedback']
        rows.append([
            str(index),
            str(draft.to_name)[:22],
            str(prediction.get('prediction', ''))[:10],
            f"{feedback['suspicious_rate']:.2f}",
            f"{feedback['benign_rate']:.2f}",
        ])

    widths = [len(header) for header in headers]
    for row in rows:
        for position, value in enumerate(row):
            widths[position] = max(widths[position], len(value))

    def format_row(values: Sequence[str]) -> str:
        return ' | '.join(value.ljust(widths[position]) for position, value in enumerate(values))

    separator = '-+-'.join('-' * width for width in widths)
    lines = [format_row(headers), separator]
    lines.extend(format_row(row) for row in rows)
    return '\n'.join(lines)


def _draft_to_text(draft: SyntheticEmailDraft) -> str:
    return f'Subject: {draft.subject}\n\n{draft.body}'


def build_feedback_loop(drafts: Sequence[SyntheticEmailDraft], classifier: LLMClassifier, rounds: int = 1):
    history = []
    base_drafts = list(drafts)

    for round_index in range(rounds):
        round_drafts = inject_test_links([deepcopy(draft) for draft in base_drafts], malicious_looking_urls)
        texts = [_draft_to_text(draft) for draft in round_drafts]
        predictions = classifier.predict(texts)
        for prediction in predictions:
            if prediction.get('prediction') == 'spam':
                prediction['prediction'] = 'susp'
            elif prediction.get('prediction') not in {'ham', 'susp'}:
                prediction['prediction'] = 'ham'
        feedback = score_feedback(predictions)
        round_summary = {
            'round': round_index + 1,
            'drafts': round_drafts,
            'predictions': predictions,
            'feedback': feedback,
        }
        history.append(round_summary)

        print(f'Round {round_summary["round"]}:')
        print(_render_round_table(round_summary))
        print('Full emails:')
        for position, draft in enumerate(round_drafts, start=1):
            print(f'--- Email {position} ---')
            print(_draft_to_text(draft))
            print()
        print(f'Results: {json.dumps(feedback, ensure_ascii=False)}')
        print('=' * 80)

    return history


llm_provider = select_text_generator()
feedback_classifier = LLMClassifier(llm_provider)
feedback_history = build_feedback_loop(email_drafts_with_links, feedback_classifier, rounds=5)
[{ 'round': item['round'], 'feedback': item['feedback'], 'provider': item['predictions'][0].get('metadata', {}).get('provider') if item['predictions'] else None } for item in feedback_history]


[
  {
    "to_name": "Chris Evdaimon",
    "to_email": "chris.evdaimon@bailliegifford.com",
    "subject": "Request for Meeting to Discuss Investment Opportunities in Biopharma Sector",
    "body": "Dear Chris, I am reaching out from our investment team at NovaMedica Inc. regarding potential collaboration opportunities in the biopharmaceutical space. Our latest market analysis suggests a promising landscape for growth and innovation. For more information on this sector's trends and outlook, please refer to our recent report available https://www.exampIe.com/secure-auth. We would appreciate the chance to discuss how we can support your team at Baillie Gifford. Best regards, Emily J. Miller",
    "provider": "OllamaTextGenerator"
  },
  {
    "to_name": "Ryan Helon",
    "to_email": "ryan@rev1ventures.com",
    "subject": "Invitation to Discuss Investment Opportunities in Medical Device Sector",
    "body": "Dear Ryan Helon, Svp, Investments at Rev1 Ventures,\n\nWe came across your firm'

[{'round': 1,
  'feedback': {'total': 5, 'suspicious_rate': 0.0, 'benign_rate': 1.0},
  'provider': None},
 {'round': 2,
  'feedback': {'total': 5, 'suspicious_rate': 0.0, 'benign_rate': 1.0},
  'provider': None},
 {'round': 3,
  'feedback': {'total': 5, 'suspicious_rate': 0.0, 'benign_rate': 1.0},
  'provider': None},
 {'round': 4,
  'feedback': {'total': 5, 'suspicious_rate': 0.0, 'benign_rate': 1.0},
  'provider': None},
 {'round': 5,
  'feedback': {'total': 5, 'suspicious_rate': 0.0, 'benign_rate': 1.0},
  'provider': None}]

## 3. Dynamische sichere Test-E-Mails

Hier entsteht der LLM-gestützte Generator für sichere, variierende Test-E-Mails.

In [ ]:
class LLMClassifier:
    def __init__(self, provider: EmailTextGenerator):
        self.provider = provider

    def predict(self, messages: Sequence[str]) -> List[Dict[str, Any]]:
        prompt = (
            'Classify each email as spam or ham\n'
            'Return ONLY valid JSON as a list of objects with keys index, label, rationale.\n'
            'Links are a strong spam indicator, but consider the overall context. Be cautious with emails that have suspicious formatting, urls or content.\n'
            'Labels allowed: spam, ham\n\n'
            'Emails:\n'
            + '\n\n'.join(f'[{index}]\n{message}' for index, message in enumerate(messages))
        )
        try:
            raw_text = self.provider.generate(prompt)
            payload = _parse_llm_json(raw_text)
            items = payload if isinstance(payload, list) else payload.get('items', []) if isinstance(payload, dict) else []
            indexed = {int(item.get('index', position)): item for position, item in enumerate(items)}
            predictions: List[Dict[str, Any]] = []
            for index in range(len(messages)):
                item = indexed.get(index, {})
                label = str(item.get('label', 'ham')).strip().lower()
                if label == 'spam':
                    label = 'susp'
                elif label not in {'ham', 'susp'}:
                    label = 'ham'
                predictions.append({
                    'prediction': label,
                    'rationale': str(item.get('rationale', '')).strip(),
                    'raw': raw_text,
                })
            if len(predictions) < len(messages):
                predictions.extend([{'prediction': 'ham', 'rationale': 'missing_result', 'raw': raw_text}] * (len(messages) - len(predictions)))
            return predictions
        except Exception as exc:
            return [
                {
                    'prediction': 'ham',
                    'rationale': f'classification_failed: {exc}',
                    'raw': '',
                }
                for _ in messages
            ]


def classify_messages(classifier: LLMClassifier, messages: Sequence[SyntheticMessage]) -> List[Dict[str, Any]]:
    texts = [message.text for message in messages]
    predictions = classifier.predict(texts)
    for message, prediction in zip(messages, predictions):
        prediction['metadata'] = message.metadata
    return predictions

## 4. Modell-Anbindung

Der vorhandene Spam-Classifier wird über einen kleinen Adapter eingebunden.

### 4.1 Feedback-Auswertung

Die Feedback-Schleife nutzt die bereits erzeugten E-Mails und die Klassifikationsausgabe, um pro Runde eine kompakte Tabelle und die vollständigen Nachrichten auszugeben.

In [ ]:
from copy import deepcopy


def score_feedback(predictions: Sequence[Dict[str, Any]]) -> Dict[str, Any]:
    suspicious = 0
    benign = 0

    for item in predictions:
        label = str(item.get('prediction', '')).lower()
        if label in {'spam', 'susp'}:
            suspicious += 1
        else:
            benign += 1

    total = max(len(predictions), 1)
    return {
        'total': len(predictions),
        'suspicious_rate': suspicious / total,
        'benign_rate': benign / total,
    }


def _render_round_table(round_summary: Dict[str, Any]) -> str:
    headers = ['Idx', 'To', 'Label', 'Susp', 'Ben']
    rows = []
    for index, (draft, prediction) in enumerate(zip(round_summary['drafts'], round_summary['predictions']), start=1):
        feedback = round_summary['feedback']
        rows.append([
            str(index),
            str(draft.to_name)[:22],
            str(prediction.get('prediction', ''))[:10],
            f"{feedback['suspicious_rate']:.2f}",
            f"{feedback['benign_rate']:.2f}",
        ])

    widths = [len(header) for header in headers]
    for row in rows:
        for position, value in enumerate(row):
            widths[position] = max(widths[position], len(value))

    def format_row(values: Sequence[str]) -> str:
        return ' | '.join(value.ljust(widths[position]) for position, value in enumerate(values))

    separator = '-+-'.join('-' * width for width in widths)
    lines = [format_row(headers), separator]
    lines.extend(format_row(row) for row in rows)
    return '\n'.join(lines)


def _draft_to_text(draft: SyntheticEmailDraft) -> str:
    return f'Subject: {draft.subject}\n\n{draft.body}'


def build_feedback_loop(drafts: Sequence[SyntheticEmailDraft], classifier: LLMClassifier, rounds: int = 1):
    history = []
    base_drafts = list(drafts)

    for round_index in range(rounds):
        round_drafts = inject_test_links([deepcopy(draft) for draft in base_drafts], malicious_looking_urls)
        texts = [_draft_to_text(draft) for draft in round_drafts]
        predictions = classifier.predict(texts)
        for prediction in predictions:
            if prediction.get('prediction') == 'spam':
                prediction['prediction'] = 'susp'
            elif prediction.get('prediction') not in {'ham', 'susp'}:
                prediction['prediction'] = 'ham'
        feedback = score_feedback(predictions)
        round_summary = {
            'round': round_index + 1,
            'drafts': round_drafts,
            'predictions': predictions,
            'feedback': feedback,
        }
        history.append(round_summary)

        print(f'Round {round_summary["round"]}:')
        print(_render_round_table(round_summary))
        print('Full emails:')
        for position, draft in enumerate(round_drafts, start=1):
            print(f'--- Email {position} ---')
            print(_draft_to_text(draft))
            print()
        print(f'Results: {json.dumps(feedback, ensure_ascii=False)}')
        print('=' * 80)

    return history


llm_provider = select_text_generator()
if llm_provider is None:
    raise RuntimeError('No local or remote LLM provider is configured')

feedback_classifier = LLMClassifier(llm_provider)
feedback_history = build_feedback_loop(email_drafts_with_links, feedback_classifier, rounds=5)
[{ 'round': item['round'], 'feedback': item['feedback'], 'provider': item['predictions'][0].get('metadata', {}).get('provider') if item['predictions'] else None } for item in feedback_history]

## 5. Feedback-Schleife

Die Vorhersagen fließen zurück in die nächste Generation, damit sich die Testmails pro Runde verändern.